# 08 MLflow Promote

This notebook applies testing and staging gates and promotes the model to production if it passes.

Note: running this notebook updates aliases and model version tags in MLflow.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [2]:
params["registry"]


{'model_name': 'IronOrePriceModel',
 'registration_output_path': 'dvc_pipeline/reports/registry_registration.json',
 'promotion_output_path': 'dvc_pipeline/reports/registry_promotion.json',
 'testing_alias': 'testing',
 'staging_alias': 'staging',
 'production_alias': 'production',
 'wait_timeout_sec': 180,
 'gates': {'testing': {'max_test_rmse': 0.5,
   'max_test_mape': 0.01,
   'min_test_r2': 0.99},
  'staging': {'max_rmse_drift_pct': 20.0, 'max_mape': 0.01, 'min_r2': 0.99}}}

In [3]:
subprocess.run([sys.executable, "dvc_pipeline/src/mlflow_promote_model.py"], check=True)


CompletedProcess(args=['c:\\aditi\\.venv\\Scripts\\python.exe', 'dvc_pipeline/src/mlflow_promote_model.py'], returncode=0)

In [4]:
read_json(params["registry"]["promotion_output_path"])


{'model_name': 'IronOrePriceModel',
 'version': '3',
 'run_id': '3d49bf2e443b44718b2539e6de052ce4',
 'final_state': 'production',
 'reason': 'passed_testing_and_staging',
 'testing_gate': {'gate': 'testing',
  'thresholds': {'max_test_rmse': 0.5,
   'max_test_mape': 0.01,
   'min_test_r2': 0.99},
  'metrics': {'test_rmse': 0.029965815851163664,
   'test_mae': 0.015143377816328898,
   'test_mape': 0.0001325401350171341,
   'test_r2': 0.9999937510939204},
  'conditions': {'rmse_ok': True, 'mape_ok': True, 'r2_ok': True},
  'passed': True},
 'staging_gate': {'gate': 'staging',
  'thresholds': {'max_rmse_drift_pct': 20.0, 'max_mape': 0.01, 'min_r2': 0.99},
  'metrics': {'rmse': 0.029965815851163768,
   'mae': 0.015143377816329124,
   'mape': 0.0001325401350171362,
   'r2': 0.9999937510939204,
   'rmse_drift_pct_vs_testing_baseline': 3.4734047981732675e-13},
  'conditions': {'rmse_drift_ok': True, 'mape_ok': True, 'r2_ok': True},
  'passed': True},
 'aliases': {'testing': 'testing',
  'stag